[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/93_assignment_sgd_minibatch.ipynb)

# Assignment 93 — Stochastic, Mini-batch, Batch Gradient Descent

Trong bài tập này, các bạn sẽ cài đặt **một** hàm gradient descent tổng quát và so sánh 3 biến thể. Học trước bài [04](04_batch_sgd_minibatch_gd.ipynb).

### Hướng dẫn làm bài
- Dùng NumPy, viết code trong `### START CODE HERE ###` … `### END CODE HERE ###`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.checks import check_close, check_equal, check_less, check_true

## 1. Xây dựng bộ dữ liệu

8 000 điểm từ phân phối chuẩn 2 chiều có tương quan cao (cột đầu là $x$, cột sau là $y$).

In [ ]:
rng = np.random.default_rng(42)
data = rng.multivariate_normal([5.0, 6.0], [[1.0, 0.95], [0.95, 1.2]], 8000)
plt.scatter(data[:500, 0], data[:500, 1], marker=".")
plt.show()

data = np.hstack((np.ones((data.shape[0], 1)), data))     # thêm cột 1 (bias)
split = int(0.90 * data.shape[0])
X_train, y_train = data[:split, :-1], data[:split, -1]
X_test, y_test = data[split:, :-1], data[split:, -1]
print("Số lượng dữ liệu train:", X_train.shape[0], "| test:", X_test.shape[0])

## 2. Hàm giả thuyết, cost và gradient

$$h_\theta(X) = X\theta, \qquad J(\theta) = \frac{1}{2m}\lVert X\theta - y\rVert^2, \qquad \nabla J(\theta) = \frac{1}{m}X^\top(X\theta - y)$$

> *So với bản gốc:* cost và gradient được **chia cho $m$** (lấy trung bình) — để learning rate có cùng ý nghĩa cho mọi kích thước batch.

In [ ]:
def hypothesis(X, theta):
    return X @ theta


def cost(X, y, theta):
    ### START CODE HERE ### (≈ 2 dòng)
    err = None
    J = None
    ### END CODE HERE ###
    return J


def gradient(X, y, theta):
    ### START CODE HERE ### (≈ 1 dòng)
    grad = None
    ### END CODE HERE ###
    return grad

In [ ]:
t = np.array([0.5, 1.0])
check_close(cost(X_train[:3], y_train[:3], t), np.mean((X_train[:3] @ t - y_train[:3]) ** 2) / 2, message="cost")
from ml_course.utils import numerical_gradient
check_close(gradient(X_train[:50], y_train[:50], t), numerical_gradient(lambda th: cost(X_train[:50], y_train[:50], th), t.copy()),
            tol=1e-6, message="gradient (gradient check)")

## 3. Gradient Descent tổng quát

Với `minibatch_size` tuỳ chọn:
```
minibatch_size = 1                 → Stochastic Gradient Descent
minibatch_size = m (số mẫu train)  → Batch Gradient Descent
1 < minibatch_size < m             → Mini-batch Gradient Descent
```
Mỗi epoch: (1) **xáo trộn** dữ liệu bằng `rng.permutation`; (2) duyệt từng batch, cập nhật $\theta := \theta - \alpha\nabla J_{\text{batch}}(\theta)$. Cuối mỗi epoch ghi lại cost trên **toàn bộ** tập train.

In [ ]:
def gradient_descent(X, y, learning_rate, minibatch_size, n_epochs=20, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros(X.shape[1])
    losses = [cost(X, y, theta)]
    start = time.perf_counter()
    for _ in range(n_epochs):
        ### START CODE HERE ### (≈ 3 dòng)
        idx = None
        X_shuffled = None
        y_shuffled = None
        ### END CODE HERE ###
        for i in range(0, len(y), minibatch_size):
            ### START CODE HERE ### (≈ 3 dòng)
            xi = None
            yi = None
            theta = None
            ### END CODE HERE ###
        losses.append(cost(X, y, theta))
    print(f"--- Thời gian chạy: {time.perf_counter() - start:.2f} giây ---")
    return theta, losses

## 4. So sánh 3 biến thể

Gọi `gradient_descent` với:
- SGD: `learning_rate = 0.005`, `minibatch_size = 1`
- Mini-batch: `learning_rate = 0.02`, `minibatch_size = 32`
- Batch: `learning_rate = 0.05`, `minibatch_size = len(y_train)`

In [ ]:
results = {}
### START CODE HERE ### (≈ 3 dòng)
pass
### END CODE HERE ###

theta_opt = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
for name, (theta, losses) in results.items():
    print(f"{name:16s} θ = {np.round(theta, 3)} | cost train = {losses[-1]:.4f} | cost test = {cost(X_test, y_test, theta):.4f}")
print(f"{'Tối ưu (lstsq)':16s} θ = {np.round(theta_opt, 3)} | cost train = {cost(X_train, y_train, theta_opt):.4f}")

plt.figure(figsize=(8, 4))
for name, (_, losses) in results.items():
    plt.plot(losses, "o-", ms=3, label=name)
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("cost trên toàn tập train")
plt.legend()
plt.show()

In [ ]:
check_equal(len(results), 3, "chạy đủ 3 biến thể")
check_less(results["Mini-batch (32)"][1][-1], 1.1 * cost(X_train, y_train, theta_opt), "mini-batch gần tối ưu sau 20 epoch")
check_true(results["SGD"][1][1] < results["Batch"][1][1], "sau 1 epoch, SGD giảm cost nhiều hơn Batch GD")

Vẽ đường hồi quy của model mini-batch trên tập test:

In [ ]:
theta_mb = results["Mini-batch (32)"][0]
plt.scatter(X_test[:, 1], y_test, marker=".")
xs = np.linspace(X_test[:, 1].min(), X_test[:, 1].max(), 50)
plt.plot(xs, theta_mb[0] + theta_mb[1] * xs, color="red")
plt.show()

### Câu hỏi suy nghĩ

1. Tính theo **epoch**, biến thể nào giảm cost nhanh nhất? Tính theo **thời gian thực**, kết quả có khác không? Vì sao?
2. Vì sao cost của SGD không giảm đều đặn như Batch GD?